# M02 Towards Monosemanticity


## Sparse autoencoder 教学实验

先用一个隐藏字典生成激活，再训练一个小型 sparse autoencoder 去恢复可复用方向。


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(11)

activation_dim = 6
true_features = 4
sae_features = 8

true_dictionary = torch.tensor(
    [
        [1.0, 0.0, 0.0, 0.5],
        [0.8, 0.1, 0.2, 0.0],
        [0.0, 0.9, 0.1, 0.4],
        [0.0, 0.8, 0.4, 0.1],
        [0.3, 0.0, 0.9, 0.2],
        [0.1, 0.2, 0.8, 1.0],
    ],
    dtype=torch.float32,
)

encoder = torch.nn.Linear(activation_dim, sae_features)
decoder = torch.nn.Linear(sae_features, activation_dim, bias=False)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.03)

for step in range(700):
    sparse_codes = (torch.rand(256, true_features) < 0.28).float() * torch.rand(256, true_features)
    activations = sparse_codes @ true_dictionary.T + 0.02 * torch.randn(256, activation_dim)
    hidden = torch.relu(encoder(activations))
    recon = decoder(hidden)
    loss = torch.nn.functional.mse_loss(recon, activations) + 0.01 * hidden.mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

decoder_weights = decoder.weight.detach().T
norms = decoder_weights.norm(dim=1)
top_indices = torch.topk(norms, k=4).indices

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(true_dictionary, cmap="viridis", aspect="auto")
axes[0].set_title("Ground-truth dictionary")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Activation dimension")

axes[1].imshow(decoder_weights[top_indices], cmap="viridis", aspect="auto")
axes[1].set_title("Recovered decoder directions (top 4)")
axes[1].set_xlabel("Activation dimension")
axes[1].set_ylabel("Recovered feature")
plt.tight_layout()

print("Top recovered feature indices:", top_indices.tolist())
print("Final loss:", float(loss.detach()))


## 小结

恢复出来的方向不会完美等于埋进去的字典，但它们比原始基底更容易复用，也更容易讲清楚。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 先选一个要扫的轴：稀疏系数、feature 数量或噪声水平。
- 在看结果前，先定义“更好的 feature recovery”是什么意思。
- 先写下哪种失败模式会让你不信任这些恢复出来的 feature。


### 运行后

- 把恢复出来的方向和真实字典对比，并指出一个不匹配点。
- 写清在你的设定里，monosemanticity 从哪里开始变差。


### 最后交付这些产物

- 1 份带 sweep 的 experiment log。
- 1 份失败边界短说明。
- 1 个下一步实验，用来区分问题更像是 capacity 还是 regularization。


## 验收题

- 和直接看 neuron 相比，SAE 恢复出的 feature 到底多给了你什么可解释性优势？
- 在你的 sweep 里，monosemanticity 是从哪里开始变差的，你认为主因更像是容量、正则化，还是数据噪声？
- 如果恢复出来的 feature 不够稳定，你下一步会先改哪个变量家族，为什么？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
